In [2]:
import pickle
import numpy as np
import pandas as pd
import os

# Define the file path (Replace with your filepath)
file_path_train = "C:/Users/Ciro/Mi Unidad (PDI)/_UGR_PDI_2024-05-30/RELIEF-T1D/Python_projects/HeterogeneityInfluence/Dataset/Preprocessed data/measurements_by_patient_train.pkl"
file_path_test = "C:/Users/Ciro/Mi Unidad (PDI)/_UGR_PDI_2024-05-30/RELIEF-T1D/Python_projects/HeterogeneityInfluence/Dataset/Preprocessed data/measurements_by_patient_test.pkl"
file_path_val = "C:/Users/Ciro/Mi Unidad (PDI)/_UGR_PDI_2024-05-30/RELIEF-T1D/Python_projects/HeterogeneityInfluence/Dataset/Preprocessed data/measurements_by_patient_val.pkl"

# Open files
with open(file_path_train, 'rb') as file:
    train_dict = pickle.load(file)

with open(file_path_test, 'rb') as file:
    test_dict = pickle.load(file)

with open(file_path_val, 'rb') as file:
    val_dict = pickle.load(file)

In [4]:
data_dict = {}
data_dict.update(train_dict)
data_dict.update(val_dict)
data_dict.update(test_dict)

In [6]:
len(data_dict)

691

In [20]:
patient_number_measurements_dict = {
    key: np.sum(~np.isnan(array))
    for key, array in data_dict.items()
}
patient_number_measurements_dict

{'LIB193263': 60023,
 'LIB193264': 26785,
 'LIB193265': 46542,
 'LIB193266': 44373,
 'LIB193267': 54578,
 'LIB193268': 41060,
 'LIB193269': 71709,
 'LIB193272': 3433,
 'LIB193273': 16157,
 'LIB193277': 114228,
 'LIB193279': 3672,
 'LIB193280': 56032,
 'LIB193284': 10292,
 'LIB193302': 27851,
 'LIB193303': 5280,
 'LIB193304': 59563,
 'LIB193308': 17200,
 'LIB193309': 77272,
 'LIB193311': 24961,
 'LIB193312': 16217,
 'LIB193313': 31153,
 'LIB193315': 31564,
 'LIB193317': 14882,
 'LIB193325': 11139,
 'LIB193326': 10283,
 'LIB193330': 11286,
 'LIB193332': 53795,
 'LIB193333': 19646,
 'LIB193334': 25988,
 'LIB193335': 20854,
 'LIB193337': 5187,
 'LIB193340': 30369,
 'LIB193341': 55481,
 'LIB193342': 64625,
 'LIB193344': 10342,
 'LIB193345': 8890,
 'LIB193346': 16208,
 'LIB193347': 36505,
 'LIB193351': 17581,
 'LIB193352': 4769,
 'LIB193354': 35085,
 'LIB193356': 62759,
 'LIB193361': 16442,
 'LIB193363': 5362,
 'LIB193367': 72654,
 'LIB193370': 8470,
 'LIB193371': 55103,
 'LIB193372': 15801,

In [8]:
import random

def make_folds_greedy(patient_measurements: dict, k=5, seed=42):
    """
    patient_measurements : dict {patient_id: n_meas}
    Devuelve una lista de k listas (IDs de pacientes por fold-test),
    con los totales de mediciones lo más iguales posible.
    """
    rng = random.Random(seed)

    # Ordena por nº de mediciones (desc) y baraja un poco para romper empates
    items = list(patient_measurements.items())
    rng.shuffle(items)
    items.sort(key=lambda x: x[1], reverse=True)

    folds = [[] for _ in range(k)]
    fold_load = [0] * k                       # mediciones acumuladas

    for pid, n in items:
        idx = fold_load.index(min(fold_load)) # fold más “ligero”
        folds[idx].append(pid)
        fold_load[idx] += n                   # actualiza su peso

    return folds


In [10]:
folds = make_folds_greedy(patient_number_measurements_dict, k=5, seed=42)

In [13]:
from typing import Dict, List
import random

Fold = Dict[str, List[str]]   # {'train': [...], 'val': [...], 'test': [...]}

def make_folds_greedy_splits(   patient_measurements: Dict[str, int],   # ← IDs como str
                                k: int = 5,
                                val_frac: float = 0.10,
                                seed: int = 42) -> List[Fold]:
    """
    Reparte pacientes (clave str) y su nº de mediciones (valor int)
    en k folds balanceando la carga total.
    Devuelve: list[ {'train': [...], 'val': [...], 'test': [...]} ]
    """
    rng = random.Random(seed)

    # 1) asignar pacientes grandes primero al fold-test más ligero
    items = list(patient_measurements.items())   # [(pid, n_meas), ...]
    rng.shuffle(items)
    items.sort(key=lambda x: x[1], reverse=True)

    test_folds: List[List[str]] = [[] for _ in range(k)]
    fold_load  = [0] * k                         # suma de mediciones por fold

    for pid, n in items:
        idx = fold_load.index(min(fold_load))
        test_folds[idx].append(pid)
        fold_load[idx] += n

    # 2) dentro de cada fold-test, crear train y val a partir del resto
    all_ids = list(patient_measurements.keys())
    folds: List[Fold] = []

    for i in range(k):
        test_ids = set(test_folds[i])
        remaining = [pid for pid in all_ids if pid not in test_ids]

        rng.shuffle(remaining)
        cut = int(val_frac * len(remaining))     # 20 % para validación
        val_ids   = remaining[:cut]
        train_ids = remaining[cut:]

        folds.append({
            'train': train_ids,
            'val'  : val_ids,
            'test' : list(test_ids)
        })

    return folds


In [36]:
from collections import Counter
from typing import Dict, List

# Type alias for clarity: each fold is a dict with three lists of patient IDs (str)
Fold = Dict[str, List[str]]      # {'train': [...], 'val': [...], 'test': [...]}

def validate_folds(
        folds: List[Fold],
        patient_measurements: Dict[str, int],
        tolerance_pct: float = 10.0) -> None:
    """
    Validate a list of patient-wise cross-validation folds.

    Parameters
    ----------
    folds : list[Fold]
        Output of `make_folds_greedy_splits` or similar.  Each element is a
        dictionary with keys ``'train'``, ``'val'``, ``'test'`` mapping to lists
        of patient IDs **(strings)**.

    patient_measurements : dict[str, int]
        Mapping ``{patient_id: number_of_measurements}``.  Used to check load
        balancing (Rule R4).

    tolerance_pct : float, default 10.0
        Maximum allowed percentage deviation from the mean test load before
        a warning is raised (Rule R4).

    Raises
    ------
    AssertionError
        If any of the mandatory rules (R1–R3) is violated.

    Side effects
    ------------
    * Prints a success message for each rule that passes.
    * Prints the test-set load per fold and the percentage deviation from the
      mean (always), and a warning if any fold exceeds *tolerance_pct*.

    Notes
    -----
    *R4* (load balance) is **informative**; it does **not** raise an error.  
    Adjust *tolerance_pct* as appropriate for your project.
    """

    all_patients = set(patient_measurements)

    # ------------------------------------------------------------------ R1/R3
    for idx, fold in enumerate(folds):
        tr, va, te = map(set, (fold['train'], fold['val'], fold['test']))

        # R1 – exclusivity within the same fold
        assert tr.isdisjoint(va) and tr.isdisjoint(te) and va.isdisjoint(te), (
            f"[R1] Overlap detected in fold {idx}"
        )

        # R3 – union covers the entire cohort
        assert len(tr | va | te) == len(all_patients), (
            f"[R3] Missing patients in fold {idx}"
        )

    print("✓ R1 and R3 passed for every fold.")

    # ------------------------------------------------------------------ R2
    test_counter = Counter(pid for fold in folds for pid in fold['test'])
    duplicates = [pid for pid, cnt in test_counter.items() if cnt != 1]

    assert not duplicates, (
        f"[R2] Patients appearing in test 0 or >1 times: {duplicates}"
    )
    print("✓ R2 passed – each patient appears in *test* exactly once.")

    # ------------------------------------------------------------------ R4 (informative)
    def test_load(ids: List[str]) -> int:
        """Total number of measurements for this group of patients."""
        return sum(patient_measurements[pid] for pid in ids)

    loads = [test_load(f['test']) for f in folds]
    mean_load = sum(loads) / len(loads)
    pct_dev = [100 * (l - mean_load) / mean_load for l in loads]

    print("\nTest-set load per fold (number of measurements):", loads)
    print("Deviation from mean (%)                      :", [round(p, 2) for p in pct_dev])

    if any(abs(p) > tolerance_pct for p in pct_dev):
        print(f"⚠️  [R4] Warning: at least one fold deviates by more than {tolerance_pct}% "
              "from the mean test load.")
    else:
        print(f"✓ R4 passed – test loads within ±{tolerance_pct}% of the mean.")

    print("\nAll mandatory validation checks completed successfully.\n")


In [26]:
folds_split = make_folds_greedy_splits(patient_number_measurements_dict, k=5, val_frac=0.2, seed=42)


In [33]:
for i, fold in enumerate(folds_split):
    print(f"\n---- Fold {i + 1}: -----")
    print(f"  Train: {len(fold['train'])} patients. Total measurements: {sum(patient_number_measurements_dict[pid] for pid in fold['train'])}")
    print(f"  Val:   {len(fold['val'])} patients. Total measurements: {sum(patient_number_measurements_dict[pid] for pid in fold['val'])}") 
    print(f"  Test:  {len(fold['test'])} patients. Total measurements: {sum(patient_number_measurements_dict[pid] for pid in fold['test'])}")
    
    print(f"\n  Train: {(len(fold['train'])/len(data_dict)*100):.2f} patients. Total measurements: {(sum(patient_number_measurements_dict[pid] for pid in fold['train'])/sum(patient_number_measurements_dict.values())*100):.2f}%")
    print(f"  Val:   {(len(fold['val'])/len(data_dict)*100):.2f} patients. Total measurements: {(sum(patient_number_measurements_dict[pid] for pid in fold['val'])/sum(patient_number_measurements_dict.values())*100):.2f}%")
    print(f"  Test:  {(len(fold['test'])/len(data_dict)*100):.2f} patients. Total measurements: {(sum(patient_number_measurements_dict[pid] for pid in fold['test'])/sum(patient_number_measurements_dict.values())*100):.2f}%")


---- Fold 1: -----
  Train: 442 patients. Total measurements: 14383391
  Val:   110 patients. Total measurements: 3387691
  Test:  139 patients. Total measurements: 4445680

  Train: 63.97 patients. Total measurements: 64.74%
  Val:   15.92 patients. Total measurements: 15.25%
  Test:  20.12 patients. Total measurements: 20.01%

---- Fold 2: -----
  Train: 443 patients. Total measurements: 14218774
  Val:   110 patients. Total measurements: 3555294
  Test:  138 patients. Total measurements: 4442694

  Train: 64.11 patients. Total measurements: 64.00%
  Val:   15.92 patients. Total measurements: 16.00%
  Test:  19.97 patients. Total measurements: 20.00%

---- Fold 3: -----
  Train: 443 patients. Total measurements: 14386280
  Val:   110 patients. Total measurements: 3387689
  Test:  138 patients. Total measurements: 4442793

  Train: 64.11 patients. Total measurements: 64.75%
  Val:   15.92 patients. Total measurements: 15.25%
  Test:  19.97 patients. Total measurements: 20.00%

---- F

In [34]:
from collections import Counter
from typing import List, Dict

def validate_folds(folds: List[Dict[str, list]],
                   patient_measurements: Dict[str, int]) -> None:
    """
    folds : salida de make_folds_greedy_splits()
    patient_measurements : {patient_id: n_meas}
    Lanza AssertionError si encuentra alguna infracción de R1-R3.
    Imprime resumen de balance (R4).
    """
    all_ids = set(patient_measurements)

    # --- R1, R3 por fold ---
    for i, fold in enumerate(folds):
        tr, va, te = map(set, (fold['train'], fold['val'], fold['test']))

        assert tr.isdisjoint(va) and tr.isdisjoint(te) and va.isdisjoint(te), \
            f"Solapamiento en fold {i}"

        assert len(tr | va | te) == len(all_ids), \
            f"Faltan pacientes en fold {i}"

    # --- R2 global ---
    test_counter = Counter(pid for fold in folds for pid in fold['test'])
    duplicates = [pid for pid, c in test_counter.items() if c != 1]
    assert not duplicates, f"Pacientes repetidos o ausentes en test: {duplicates}"

    # --- R4 balance opcional ---
    def fold_load(ids):        # nº de mediciones de este grupo
        return sum(patient_measurements[pid] for pid in ids)

    test_loads = [fold_load(f['test']) for f in folds]
    print("Carga (mediciones) por fold-test:", test_loads)
    print("Desviación % respecto a la media:",
          [round(100*(l - sum(test_loads)/len(test_loads)) / (sum(test_loads)/len(test_loads)),2)
           for l in test_loads])


In [35]:
validate_folds(folds_split, patient_number_measurements_dict)

Carga (mediciones) por fold-test: [4445680, 4442694, 4442793, 4442808, 4442787]
Desviación % respecto a la media: [0.05, -0.01, -0.01, -0.01, -0.01]
